#**Purpose**

This Notebook runs inference on the fine tuned TinyLlama 1.1B Chat model for MCQ question-answer-pair generation to see whether the model is working properly or not.

## **Install the required modules**

In [2]:
!pip install -qU \
transformers\
torch\
datasets\
torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 56.3 MB/s eta 0:00:00


**Restart the session after all the modules are installed.**

##**Import the required Libraries**

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from google.colab import drive

##**Initialize the Model from Google Drive**

In [5]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Path where your merged model is saved on Google Drive
merged_model_path = "/content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq"

print("Loading tokenizer and fine-tuned merged model from Google Drive...")
tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
model = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

Mounted at /content/drive
Loading tokenizer and fine-tuned merged model from Google Drive...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

###**Initialize the text-generation pipeline**

In [6]:
qg_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=128,
    do_sample=False  # Deterministic generation for evaluation benchmarks
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


###**Interactive Testing Function**

In [7]:
def generate_mcq_question(context, target_answer):
    messages = [
        {
            "role": "system",
            "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
        },
        {
            "role": "user",
            "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = qg_pipeline(prompt)
    return response[0]['generated_text']

####**Testing with First Sample**

In [8]:
test_context = "The Transformer architecture relies entirely on attention mechanisms to draw global dependencies between input and output, eschewing recurrence and convolutions entirely."
test_answer = "attention mechanisms"

print(generate_mcq_question(test_context, test_answer))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question: What is the Transformer architecture's primary mechanism for drawing global dependencies between input and output?
Answer: attention mechanisms


####**Testing with Second Sample**

In [9]:
test_context = "Electromagnetic induction is the production of an electromotive force across an electrical conductor in a changing magnetic field. Faraday's law of induction predicts how a magnetic field will interact with an electric circuit to produce an EMF."
test_answer = "Faraday's law"

print(generate_mcq_question(test_context, test_answer))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What law predicts how a magnetic field will interact with an electric circuit to produce an EMF?
Answer: Faraday's law
